# LAB03 — Zero-Shot vs Few-Shot Prompting & Example Selection

## Amac

Bu lab, **Zero-Shot Prompting** ile **Few-Shot Prompting** (In-Context
Learning) arasindaki farki ve Few-Shot Example seciminin (ornek sayisi,
cesitliligi, temsil gucu, siralamasi) model performansi uzerindeki etkisini
deneysel olarak inceler. Hayali bir e-ticaret sirketinin **Customer Support
Assistant** uygulamasi icin, gelen musteri mesajlarini bes kategoriden
birine siniflandiriyoruz:

```text
billing, technical_issue, cancellation, complaint, general_inquiry
```

Bu lab'de **sadece Category Classification** yapilir; LAB01'deki Priority
Classification bu lab'e dahil edilmemistir (amac, Example Selection
etkisini mumkun oldugunca bagimsiz olcebilmektir).

## Temel Kavramlar

- **Zero-Shot Prompting**: Modele gorevi yalnizca talimatlarla (instruction,
  kategori tanimlari, karar kurallari, cikti sozlesmesi) aciklamak; hicbir
  ornek giris-cikis cifti gostermemek.
- **Few-Shot Prompting / In-Context Learning**: Modele, gercek soruyu
  sormadan once, birkac ornek giris-cikis cifti (`user`/`assistant`
  mesaj turleri) gostermek. Bu, modelin agirliklarini guncelleyen bir
  fine-tuning islemi **DEGILDIR** — ornekler sadece o tek istek icin
  baglam (context) olarak sunulur.
- **Example Quality / Diversity / Representativeness**: Bir few-shot
  ornegin dogru etiketlenmis olmasi (quality) ile orneklerin birbirinden
  farkli olmasi (diversity) ve tum kategorileri dengeli temsil etmesi
  (representativeness) birbirinden ayri kavramlardir.
- **Example Selection & Ordering**: Hangi orneklerin secildigi ve bu
  orneklerin modele hangi sirada sunuldugu, modelin kararini etkileyebilir.
- **Token Cost vs Accuracy Trade-off**: Few-shot ornekler prompt token
  sayisini artirir; bu artisin dogruluga gercekten yansiyip yansimadigi
  bu lab'de olculur, varsayilmaz.

## Arastirma Sorulari

Bunlar **arastirma sorularidir**, varsayim degildir. Modelin hangi deneyde
daha basarili olacagini onceden tahmin etmeyin; asagidaki sorulari yalnizca
gercek olcumlerle (Step 10-12) cevaplayin.

1. Zero-Shot Prompting hangi durumlarda yeterli olabilir?
2. Few-Shot Examples eklemek her zaman Accuracy artirir mi?
3. Birbirine benzeyen (redundant) ornekler ile farkli kategorileri temsil
   eden (diverse) ornekler arasinda performans farki var mi?
4. Orneklerin siralamasini degistirmek model kararlarini etkiliyor mu?
5. Daha fazla prompt token tuketmek daha iyi siniflandirma anlamina
   geliyor mu?

## Deneyler

| Experiment | Example Count | Selection Strategy | Ordering |
|---|---:|---|---|
| `E01_zero_shot` | 0 | Ornek yok | N/A |
| `E02_redundant_few_shot` | 5 | Taraflı / tekrarlayan (4 billing + 1 technical_issue) | Sabit |
| `E03_diverse_few_shot` | 5 | Secilmis / dengeli (5 kategoriden birer ornek) | Sabit |
| `E04_reversed_order` | 5 | E03 ile birebir ayni ornekler | Ters cevrilmis |

**Not:** Bu notebook acildiginda hicbir ucretli API cagrisi otomatik olarak
yapilmaz. Gercek cagrilar yalnizca `RUN_LIVE = True` yapildiginda
gerceklesir.


## Step 2 — Configuration

Model parametrelerini (model adi, temperature, max tokens) LAB01/LAB02 ile
**ayni** `config/settings.toml` dosyasindan okuyoruz; bu dosyaya hicbir
yazma islemi yapmiyoruz. Prompt bilesenlerini ve deney tanimlarini ise
LAB03'e ozel `config/lab03_prompts.yaml` dosyasindan okuyoruz.

Deney ayarlarini (`LAB03_MAX_CASES`, `LAB03_REPETITIONS`) ise, LAB01 ve
LAB02'nin `[experiment]` ayarlarini etkilememek icin **notebook icinde**,
`config/settings.toml`'dan bagimsiz olarak tanimliyoruz.


In [ ]:
import os
import json
import time
import re
from pathlib import Path

import tomli
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "config").exists() else CWD.parent

with open(PROJECT_ROOT / "config" / "settings.toml", "rb") as f:
    settings = tomli.load(f)

with open(PROJECT_ROOT / "config" / "lab03_prompts.yaml", "r", encoding="utf-8") as f:
    lab03_prompts = yaml.safe_load(f)

MODEL_NAME = settings["model"]["name"]
MODEL_TEMPERATURE = settings["model"]["temperature"]
MODEL_MAX_TOKENS = settings["model"]["max_completion_tokens"]

# LAB03'e ozel deney ayarlari. Kasitli olarak config/settings.toml'daki
# [experiment] bolumunden OKUNMAZ; boylece LAB03 degisiklikleri LAB01 ve
# LAB02'nin deney parametrelerini etkilemez.
LAB03_MAX_CASES = 10
LAB03_REPETITIONS = 1

print("Model ayarlari (LAB01/LAB02 ile ayni config/settings.toml):")
print(f"  provider               : {settings['model']['provider']}")
print(f"  name                   : {MODEL_NAME}")
print(f"  temperature            : {MODEL_TEMPERATURE}")
print(f"  max_completion_tokens  : {MODEL_MAX_TOKENS}")
print()
print("LAB03'e ozel deney ayarlari (config/settings.toml'dan bagimsiz):")
print(f"  LAB03_MAX_CASES    : {LAB03_MAX_CASES}")
print(f"  LAB03_REPETITIONS  : {LAB03_REPETITIONS}")
print()
print("LAB03 deneyleri:", list(lab03_prompts["experiments"].keys()))


## Step 3 — Dataset Exploration

Iki ayri veri seti kullaniyoruz:

- `data/lab03_evaluation.json`: Deneylerde siniflandirilacak 20 sentetik
  musteri mesaji (her kategoriden 4 tane), beklenen kategori, zorluk
  seviyesi ve kisa bir not iceriyor.
- `data/lab03_examples.json`: Few-Shot Prompting icin kullanilacak 15
  aday ornek. Bu ornekler **evaluation dataset'inde yer almaz** (data
  leakage onlemi) — asagida bunu programatik olarak da dogruluyoruz.

`LAB03_MAX_CASES = 10` oldugunda, her kategoriden 2 vaka secilerek 5
kategorinin tamami temsil edilsin istiyoruz. Secim, veri setindeki orijinal
sirayi (dolayisiyla `id` sirasini) koruyarak yapilir; bu nedenle
tekrarlanabilir ve deterministiktir.


In [ ]:
with open(PROJECT_ROOT / "data" / "lab03_evaluation.json", "r", encoding="utf-8") as f:
    evaluation_data = json.load(f)

with open(PROJECT_ROOT / "data" / "lab03_examples.json", "r", encoding="utf-8") as f:
    examples_data = json.load(f)

df_eval_all = pd.DataFrame(evaluation_data)
df_examples = pd.DataFrame(examples_data)
examples_by_id = {example["id"]: example for example in examples_data}

print(f"Evaluation dataset boyutu : {len(df_eval_all)}")
print("Evaluation kategori dagilimi:")
print(df_eval_all["expected_label"].value_counts())
print()
print(f"Example candidate pool boyutu: {len(df_examples)}")
print("Example pool kategori dagilimi:")
print(df_examples["label"].value_counts())


In [ ]:
# Data Leakage kontrolu: evaluation dataset'indeki hicbir mesaj, example
# pool'da (kelimesi kelimesine) yer almamali. Bu basit, tam-eslesme (exact
# match) tabanli bir kontroldur; kucuk kelime degisiklikleriyle yazilmis
# near-duplicate'lar bu kontrolden GECEBILIR, bu nedenle veri setleri
# elle (manuel) olarak da farkli ifadelerle hazirlanmistir.
eval_messages = set(df_eval_all["message"].str.strip().str.lower())
example_messages = set(df_examples["message"].str.strip().str.lower())
overlap = eval_messages & example_messages

assert len(overlap) == 0, f"Data leakage tespit edildi: {overlap}"
assert len(df_eval_all) == 20, "Evaluation dataset tam olarak 20 kayit icermeli."
assert df_eval_all["expected_label"].value_counts().nunique() == 1, "Her kategori esit sayida temsil edilmeli."
assert len(df_examples) >= 12, "Example pool en az 12 aday ornek icermeli."

print("Dogrulandi: Data leakage yok, evaluation dataset 20 kayit, kategoriler esit dagilimli.")


In [ ]:
CATEGORY_ORDER = ["billing", "technical_issue", "cancellation", "complaint", "general_inquiry"]
N_CATEGORIES = len(CATEGORY_ORDER)
PER_CATEGORY = max(1, LAB03_MAX_CASES // N_CATEGORIES)

# Basit ve deterministik secim: her kategoriden (dosyadaki orijinal sirayi
# koruyarak) PER_CATEGORY kadar vaka al. df_eval_all zaten id sirasina
# gore (ve kategoriler bloklar halinde) siralandigi icin, filtrelemek
# orijinal sirayi dogal olarak korur.
selected_ids = []
for label in CATEGORY_ORDER:
    label_ids = df_eval_all.loc[df_eval_all["expected_label"] == label, "id"].head(PER_CATEGORY).tolist()
    selected_ids.extend(label_ids)

eval_cases = df_eval_all[df_eval_all["id"].isin(selected_ids)].reset_index(drop=True)

print(f"LAB03_MAX_CASES={LAB03_MAX_CASES}, kategori basina secilen vaka sayisi={PER_CATEGORY}")
print(f"Secilen toplam vaka sayisi: {len(eval_cases)}")
print("Secilen vakalarin kategori dagilimi:")
print(eval_cases["expected_label"].value_counts())
eval_cases


## Step 4 — Zero-Shot vs Few-Shot Concepts

**Zero-Shot (`E01_zero_shot`)**: Modele yalnizca `system` mesaji icinde
identity, instruction, category definitions, classification rules ve
output contract verilir. Hicbir ornek giris-cikis cifti yoktur. Model,
gorevi tamamen talimat metninden "anlayarak" cozmek zorundadir.

**Few-Shot (`E02`-`E04`)**: Ayni `system` mesajina ek olarak, gercek soruyu
icerecek `user` mesajindan ONCE, her biri bir `user` (ornek mesaj) ve
`assistant` (dogru `ETIKET: ...` yaniti) cifti olan birkac "ornek turu"
eklenir. Bu, OpenAI/Groq tarzi chat completion API'lerinde Few-Shot /
In-Context Learning'in tipik temsil bicimidir: ornekler, modelin daha once
"gordugu" bir konusma gibi sunulur, ama hicbir model agirligi
guncellenmez — ornekler sadece bu TEK istek icin gecerlidir.

Asagida, her iki yaklasimin aynisiniflandirma sorusu icin urettigi
`messages` dizisini yan yana karsilastiracagiz (bkz. Step 6 ve Step 7).


## Step 5 — Example Selection Strategies

Few-shot ornekler secilirken dikkat edilmesi gereken birkac bagimsiz
boyut vardir:

- **Redundancy (Tekrarlilik)**: Orneklerin birbirine cok benzemesi veya
  belirli bir kategoride yogunlasmasi (`E02_redundant_few_shot`: 4 billing
  + 1 technical_issue). Dusuk cesitlilik, modelin diger kategorileri
  "gormeden" karar vermesine yol acabilir.
- **Diversity (Cesitlilik) & Representativeness (Temsil Gucu)**:
  Orneklerin farkli kategorileri dengeli sekilde temsil etmesi
  (`E03_diverse_few_shot`: 5 kategoriden birer ornek).
- **Example Quality**: Ornegin dogru etiketlenmis olmasi. Bu lab'de TUM
  ornekler dogru etiketlidir — kasitli olarak yanlis etiketlenmis ornek
  kullanilmaz. Quality ile diversity birbirinden tamamen ayri kavramlardir;
  dusuk cesitlilikli bir ornek kumesi (E02) yine de dogru etiketli
  olabilir.
- **Example Ordering & Order Sensitivity**: Ayni ornek kumesinin modele
  hangi sirada sunuldugu (`E03` vs `E04`, orneklerin tersine cevrilmis
  hali).

**Metodolojik sinirlama:** `E02` ile `E03` arasinda yalnizca diversity
degil, orneklerin icerikleri ve temsil gucu de birlikte degismektedir. Bu
nedenle aralarindaki performans farkini tek basina diversity'ye baglamak
dogru degildir — bu iki deney BIRLIKTE farkli bir example selection
stratejisini temsil eder.


## Step 6 — Prompt Construction

`build_messages()` fonksiyonu, bir deney (`experiment_key`) ve bir vaka
icin tam `messages` dizisini olusturur:

1. `system` mesaji: identity + instruction + category_definitions +
   classification_rules + output_contract (butun deneylerde AYNI).
2. Her few-shot example icin bir `user` (`"Musteri mesaji: ..."`) ve bir
   `assistant` (`"ETIKET: <label>"`) mesaj cifti — `config/lab03_prompts.yaml`
   icindeki `experiments.<key>.example_ids` sirasiyla.
3. Gercek siniflandirilacak vakanin `user` mesaji (her zaman en sonda).

Bu yapi, `E04_reversed_order` icin yalnizca 2. adimdaki ornek turlerinin
sirasini degistirmemizi; 1. ve 3. adimlari hic degistirmememizi saglar.


In [ ]:
def build_messages(experiment_key, case):
    experiment = lab03_prompts["experiments"][experiment_key]

    system_parts = [
        lab03_prompts["identity"].strip(),
        lab03_prompts["instruction"].strip(),
        lab03_prompts["category_definitions"].strip(),
        lab03_prompts["classification_rules"].strip(),
        lab03_prompts["output_contract"].strip(),
    ]
    system_message = "\n\n".join(system_parts)

    template = lab03_prompts["user_message_template"]
    messages = [{"role": "system", "content": system_message}]

    for example_id in experiment["example_ids"]:
        example = examples_by_id[example_id]
        messages.append({"role": "user", "content": template.format(message=example["message"])})
        messages.append({"role": "assistant", "content": f"ETIKET: {example['label']}"})

    messages.append({"role": "user", "content": template.format(message=case["message"])})
    return messages


LAB03_EXPERIMENT_KEYS = list(lab03_prompts["experiments"].keys())
print("Tanimli deneyler:", LAB03_EXPERIMENT_KEYS)


In [ ]:
# Deney tasarimi dogrulamalari (Section 17 gereksinimleri, Groq API cagrisi yapilmaz)
assert len(lab03_prompts["experiments"]["E01_zero_shot"]["example_ids"]) == 0, "E01'de hicbir few-shot example bulunmamali."
assert len(lab03_prompts["experiments"]["E02_redundant_few_shot"]["example_ids"]) == 5, "E02'de tam 5 example olmali."
assert len(lab03_prompts["experiments"]["E03_diverse_few_shot"]["example_ids"]) == 5, "E03'te tam 5 example olmali."

e03_ids = lab03_prompts["experiments"]["E03_diverse_few_shot"]["example_ids"]
e04_ids = lab03_prompts["experiments"]["E04_reversed_order"]["example_ids"]
assert sorted(e03_ids) == sorted(e04_ids), "E04, E03 ile AYNI orneklerin kullanmali."
assert e04_ids == list(reversed(e03_ids)), "E04, E03'un orneklerini TERS sirada kullanmali."
assert e03_ids != e04_ids, "Siralama gercekten degismeli (ayni sirada degil)."

print("Dogrulandi: E01 ornek icermiyor, E02/E03 tam 5 example iceriyor, E04 = reverse(E03).")


In [ ]:
# Her deneyin aynı vaka icin urettigi messages dizisini karsilastiralim.
# Ozellikle E01 (zero-shot) ile E03 (diverse few-shot) arasindaki farka
# dikkat edin: E03'un messages dizisi, gercek soru oncesinde 5 ek
# user/assistant cifti icerir.
sample_case = eval_cases.iloc[0]

for key in LAB03_EXPERIMENT_KEYS:
    experiment = lab03_prompts["experiments"][key]
    messages = build_messages(key, sample_case)
    print(f"===== {key} ({experiment['name']}) — {len(messages)} messages, {len(experiment['example_ids'])} example(s) =====")
    for message in messages:
        preview = message["content"].replace("\n", " ")
        if len(preview) > 100:
            preview = preview[:100] + "..."
        print(f"  [{message['role']}] {preview}")
    print()


## Step 7 — Groq Client & Request JSON Inspection

LAB01/LAB02 ile ayni yaklasim: `RUN_LIVE` degiskeni `False` oldugu surece
hicbir gercek API cagrisi yapilmaz.

Iki goruntuleme kontrolu, API cagrisini **ACIP KAPATMAZ**; yalnizca
notebook'ta ne gosterildigini kontrol eder:

- `SHOW_REQUEST_JSON`: `True` ise, Groq SDK'ya gonderilen request payload'i
  (`model`, `temperature`, `max_completion_tokens`, `messages`) JSON olarak
  yazdirilir. API key veya herhangi bir secret bu JSON'da **yer almaz**.
- `SHOW_ALL_REQUESTS`: `False` ise her deney icin yalnizca **ilk** request
  gosterilir; `True` ise butun request'ler gosterilir.

**E01 ile E03'u karsilastirin:** E01'in `messages` listesi `[system, user]`
seklindedir; E03'unki ise `[system, user, assistant, user, assistant, ...,
user]` seklindedir — few-shot ornekler, `messages` dizisine ek
user/assistant turleri olarak eklenir, system talimatinin kendisi
degismez.


In [ ]:
RUN_LIVE = False
SHOW_REQUEST_JSON = True
SHOW_ALL_REQUESTS = False

api_key = os.getenv("GROQ_API_KEY")

if RUN_LIVE and not api_key:
    raise RuntimeError(
        "GROQ_API_KEY bulunamadi. Proje kokunde bir '.env' dosyasi olusturup "
        "GROQ_API_KEY=... satirini ekleyin (.env.example dosyasina bakabilirsiniz)."
    )

client = Groq(api_key=api_key) if api_key else None

print(f"RUN_LIVE          : {RUN_LIVE}")
print(f"SHOW_REQUEST_JSON  : {SHOW_REQUEST_JSON}")
print(f"SHOW_ALL_REQUESTS  : {SHOW_ALL_REQUESTS}")
print(f"API key bulundu    : {api_key is not None}")


In [ ]:
# E01 (zero-shot) ve E03 (diverse few-shot) icin ayni vakanin request
# JSON'unu yan yana yazdiriyoruz. Gercek API cagrisi YAPILMAZ.
for key in ["E01_zero_shot", "E03_diverse_few_shot"]:
    request_payload = {
        "model": MODEL_NAME,
        "temperature": MODEL_TEMPERATURE,
        "max_completion_tokens": MODEL_MAX_TOKENS,
        "messages": build_messages(key, sample_case),
    }
    print(f"----- Request JSON: {key} (case_id={sample_case['id']}) -----")
    print(json.dumps(request_payload, ensure_ascii=False, indent=2))
    print()


## Step 8 — Execute Experiments

**Ne test ediyoruz?** Dort farkli few-shot example stratejisinin (yok /
taraflı / dengeli / dengeli-ters-sirali) ayni siniflandirma gorevindeki
etkisini.

**Ne degisiyor?** Yalnizca few-shot example secimi (`E02` vs `E03`) ve
siralamasi (`E03` vs `E04`).

**Ne sabit kaliyor?** Model, temperature, max tokens, system talimati
(identity, instruction, category definitions, classification rules,
output contract), evaluation dataset, vaka sirasi ve repetitions —
bkz. Section 9 (Controlled Experiment Requirements). Her deney, Step 3'te
secilen AYNI `eval_cases` kumesini, AYNI sirada degerlendirir.

Bu adimda model yanitlarini **ham** (henuz ayristirilmamis) olarak
saklariz; ayristirma (parsing) ve format kontrolu Step 9'da yapilir — bu
iki islemi kasitli olarak ayri tutuyoruz.


In [ ]:
def run_experiment(experiment_key, cases, repetitions):
    results = []
    requests_shown = 0
    for _, case in cases.iterrows():
        for repetition in range(1, repetitions + 1):
            messages = build_messages(experiment_key, case)
            request_payload = {
                "model": MODEL_NAME,
                "temperature": MODEL_TEMPERATURE,
                "max_completion_tokens": MODEL_MAX_TOKENS,
                "messages": messages,
            }

            if SHOW_REQUEST_JSON and (SHOW_ALL_REQUESTS or requests_shown == 0):
                print(f"----- Request JSON: {experiment_key} | case_id={case['id']} | repetition={repetition} -----")
                print(json.dumps(request_payload, ensure_ascii=False, indent=2))
                print()
            requests_shown += 1

            start = time.perf_counter()
            try:
                completion = client.chat.completions.create(**request_payload)
                latency_ms = (time.perf_counter() - start) * 1000
                response_text = completion.choices[0].message.content
                usage = completion.usage
                results.append({
                    "experiment": experiment_key,
                    "case_id": case["id"],
                    "repetition": repetition,
                    "expected_label": case["expected_label"],
                    "difficulty": case["difficulty"],
                    "message": case["message"],
                    "model_response": response_text,
                    "prompt_tokens": usage.prompt_tokens,
                    "completion_tokens": usage.completion_tokens,
                    "total_tokens": usage.total_tokens,
                    "latency_ms": latency_ms,
                    "error": None,
                })
            except Exception as exc:
                latency_ms = (time.perf_counter() - start) * 1000
                print(f"[HATA] experiment={experiment_key} case_id={case['id']} repetition={repetition}: {exc}")
                results.append({
                    "experiment": experiment_key,
                    "case_id": case["id"],
                    "repetition": repetition,
                    "expected_label": case["expected_label"],
                    "difficulty": case["difficulty"],
                    "message": case["message"],
                    "model_response": None,
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                    "latency_ms": latency_ms,
                    "error": str(exc),
                })
    return results


In [ ]:
# Gercek API cagrisi yapmadan, RUN_LIVE'dan BAGIMSIZ olarak kac istek
# gonderilecegini hesaplayip gosteriyoruz.
n_cases = len(eval_cases)
request_counts = {key: n_cases * LAB03_REPETITIONS for key in LAB03_EXPERIMENT_KEYS}
total_live_requests = sum(request_counts.values())

for key, count in request_counts.items():
    print(f"{key} Requests: {count}")
print(f"Total Live API Requests: {total_live_requests}")


In [ ]:
RESULT_COLUMNS = [
    "experiment", "case_id", "repetition", "expected_label", "difficulty", "message",
    "model_response", "prompt_tokens", "completion_tokens", "total_tokens", "latency_ms", "error",
]

if RUN_LIVE:
    if client is None:
        raise RuntimeError("Groq client baslatilamadi; GROQ_API_KEY eksik.")

    all_results = []
    for experiment_key in LAB03_EXPERIMENT_KEYS:
        all_results.extend(run_experiment(experiment_key, eval_cases, LAB03_REPETITIONS))
    df_results = pd.DataFrame(all_results, columns=RESULT_COLUMNS)
else:
    print("RUN_LIVE = False oldugu icin gercek API cagrisi yapilmadi.")
    print("Bu hucreyi calistirmadan once Step 9-11'i bos sonuclarla inceleyebilirsiniz.")
    df_results = pd.DataFrame(columns=RESULT_COLUMNS)

df_results


## Step 9 — Response Parsing & Validation

Iki ayri kontrol fonksiyonu tanimliyoruz; bu ikisinin amaci **farklidir**:

- `parse_label()`: Model yanitindan siniflandirma etiketini **cikarir**
  (serbest formatli yanitlar da dahil). Bu, **classification accuracy**
  icin kullanilir ve output contract'a tam uyumdan bagimsizdir.
- `is_format_compliant()`: Yanitin, output contract'a (`ETIKET: <label>`,
  TAM OLARAK tek satir, baska hicbir ek metin olmadan) **harfiyen uyup
  uymadigini** kontrol eder. Bir yanit dogru siniflandirilmis olabilir ama
  bu sozlesmeye uymayabilir (ornegin aciklama ekleyerek); bunun tersi
  gecerli degildir, cunku sozlesmeye uyan her yanit gecerli bir etiket de
  icermek zorundadir.


In [ ]:
VALID_LABELS = {"billing", "technical_issue", "cancellation", "complaint", "general_inquiry"}


def parse_label(response_text):
    """Serbest formatli model yanitlarindan siniflandirma etiketini cikarir.

    Kelime sinirlariyla eslesme yapar (substring eslesmesi yapmaz, ornegin
    'not_billing' icinde 'billing' aranmaz). Birden fazla farkli etiket
    eslesirse (celiskili yanit) None doner.
    """
    if not response_text:
        return None
    pattern = r"\b(" + "|".join(re.escape(label) for label in VALID_LABELS) + r")\b"
    matches = re.findall(pattern, response_text, flags=re.IGNORECASE)
    unique_matches = {m.lower() for m in matches}
    return unique_matches.pop() if len(unique_matches) == 1 else None


def is_format_compliant(response_text):
    """Yanitin TAM OLARAK tek satirlik ETIKET sozlesmesine uyup uymadigini kontrol eder.

    parse_label'dan tamamen bagimsizdir: dogru siniflandirilmis ama
    formata uymayan yanitlar burada False doner.
    """
    if not response_text:
        return False
    lines = [line.strip() for line in response_text.strip().splitlines()]
    if len(lines) != 1:
        return False
    match = re.fullmatch(r"ETIKET:\s*(\w+)", lines[0], flags=re.IGNORECASE)
    if not match:
        return False
    return match.group(1).lower() in VALID_LABELS


In [ ]:
# Parser ve format compliance dogrulama testleri (Groq API cagrisi yapilmaz)
assert parse_label("ETIKET: billing") == "billing"  # valid strict format
assert is_format_compliant("ETIKET: billing") is True

assert parse_label("billing") == "billing"  # valid category-only response
assert is_format_compliant("billing") is False

assert parse_label("ETIKET: technical_issue") == "technical_issue"
assert is_format_compliant("ETIKET: technical_issue") is True

assert parse_label("unknown_category") is None  # invalid category
assert is_format_compliant("unknown_category") is False

assert parse_label("") is None  # empty response
assert is_format_compliant("") is False

assert parse_label(None) is None  # None response
assert is_format_compliant(None) is False

assert parse_label("Bu mesaj billing ya da complaint olabilir") is None  # conflicting categories

assert parse_label("ETIKET: billing\nAciklama: ek bilgi") == "billing"  # extra text, parser still finds it
assert is_format_compliant("ETIKET: billing\nAciklama: ek bilgi") is False  # strict format rejects it

assert parse_label("not_billing") is None  # category name embedded in a longer identifier

print("Tum parser ve format compliance testleri basarili.")


In [ ]:
if not df_results.empty:
    df_results["parsed_label"] = df_results["model_response"].apply(parse_label)
    df_results["format_compliant"] = df_results["error"].isna() & df_results["model_response"].apply(is_format_compliant)
    df_results["label_correct"] = df_results["error"].isna() & (df_results["parsed_label"] == df_results["expected_label"])
    df_results["is_api_error"] = df_results["error"].notna()
    df_results["is_invalid"] = df_results["error"].isna() & df_results["parsed_label"].isna()
else:
    print("Henuz sonuc yok. Once Step 8'de RUN_LIVE = True yapip deneyleri calistirin.")

df_results


In [ ]:
# Hatali (API error) ve basarili-ama-ayristirilamayan (invalid format) yanitlari
# AYRI goruntuluyoruz; bir API hatasi dogrudan "modelin gecersiz format
# urettigi" anlamina gelmez.
if not df_results.empty:
    df_api_errors = df_results[df_results["is_api_error"]]
    df_unparseable = df_results[df_results["is_invalid"]]
    print(f"API hatasi alan cagri sayisi                                : {len(df_api_errors)}")
    print(f"Basarili cagri ama ayristirilamayan (gecersiz etiket) yanit sayisi: {len(df_unparseable)}")
    df_results[df_results["is_api_error"] | df_results["is_invalid"]]
else:
    print("Henuz sonuc yok.")


## Step 10 — Evaluation Metrics

Her deney icin su metrikleri hesapliyoruz:

**Classification Metrics:** `n_cases`, `n_repetitions`, `n_success`,
`api_error_rate`, `label_accuracy`, `invalid_response_rate`,
`format_compliance_rate`.

**Token & Performance Metrics:** `avg_prompt_tokens`,
`avg_completion_tokens`, `avg_total_tokens`, `avg_latency_ms`.

Hatali veya ayristirilamayan yanitlar, dogruluk ve format kontrollerinde
otomatik olarak basarisiz/`False` sayilir; sessizce basarili kabul edilmez.

Ayrica **Example Selection Analysis** olarak her deneyin example sayisini,
example kategori dagilimini, kategori bazli (per-category) accuracy'sini
ve yanlis siniflandirilan vakalari raporluyoruz; son olarak `E03` ile `E04`
arasinda hangi vakalarda model kararinin degistigini ayrica gosteriyoruz.


In [ ]:
if not df_results.empty:
    df_metrics = df_results.groupby("experiment").agg(
        n_cases=("case_id", "count"),
        n_repetitions=("repetition", "max"),
        n_success=("error", lambda s: s.isna().sum()),
        api_error_rate=("is_api_error", "mean"),
        label_accuracy=("label_correct", "mean"),
        invalid_response_rate=("is_invalid", "mean"),
        format_compliance_rate=("format_compliant", "mean"),
        avg_prompt_tokens=("prompt_tokens", "mean"),
        avg_completion_tokens=("completion_tokens", "mean"),
        avg_total_tokens=("total_tokens", "mean"),
        avg_latency_ms=("latency_ms", "mean"),
    )
    df_metrics = df_metrics.reindex(LAB03_EXPERIMENT_KEYS)
else:
    print("Henuz sonuc yok. Once Step 8'de RUN_LIVE = True yapip deneyleri calistirin.")
    df_metrics = pd.DataFrame()

df_metrics


In [ ]:
if not df_metrics.empty:
    df_comparison = df_metrics.copy()
    df_comparison.index.name = "experiment"
    df_comparison = df_comparison.reset_index()
    df_comparison["experiment_name"] = df_comparison["experiment"].map(
        lambda key: lab03_prompts["experiments"][key]["name"]
    )
    df_comparison["example_count"] = df_comparison["experiment"].map(
        lambda key: len(lab03_prompts["experiments"][key]["example_ids"])
    )
else:
    print("Karsilastirma tablosu icin once deneyleri calistirin (Step 8).")
    df_comparison = pd.DataFrame()

if not df_comparison.empty:
    display_table = df_comparison.style.format({
        "api_error_rate": "{:.0%}",
        "label_accuracy": "{:.0%}",
        "invalid_response_rate": "{:.0%}",
        "format_compliance_rate": "{:.0%}",
        "avg_prompt_tokens": "{:.1f}",
        "avg_completion_tokens": "{:.1f}",
        "avg_total_tokens": "{:.1f}",
        "avg_latency_ms": "{:.1f}",
    })
else:
    display_table = df_comparison

display_table


In [ ]:
# Example Selection Analysis: her deneyin example sayisi ve kategori dagilimi.
if not df_results.empty:
    example_distribution_rows = []
    for key in LAB03_EXPERIMENT_KEYS:
        example_ids = lab03_prompts["experiments"][key]["example_ids"]
        labels = [examples_by_id[eid]["label"] for eid in example_ids]
        distribution = pd.Series(labels).value_counts().to_dict() if labels else {}
        example_distribution_rows.append({
            "experiment": key,
            "example_count": len(example_ids),
            "example_category_distribution": distribution if distribution else "N/A (zero-shot)",
        })
    df_example_distribution = pd.DataFrame(example_distribution_rows)
else:
    print("Henuz sonuc yok.")
    df_example_distribution = pd.DataFrame()

df_example_distribution


In [ ]:
# Per-Category Accuracy by Experiment
if not df_results.empty:
    df_per_category_accuracy = (
        df_results.groupby(["experiment", "expected_label"])["label_correct"]
        .mean()
        .unstack("expected_label")
        .reindex(LAB03_EXPERIMENT_KEYS)
        .reindex(columns=CATEGORY_ORDER)
    )
else:
    print("Henuz sonuc yok.")
    df_per_category_accuracy = pd.DataFrame()

df_per_category_accuracy


In [ ]:
# Incorrectly Classified Cases: hatasiz cagrilar arasinda yanlis siniflandirilanlar.
if not df_results.empty:
    df_incorrect = df_results[
        df_results["error"].isna() & ~df_results["label_correct"]
    ][["experiment", "case_id", "expected_label", "parsed_label", "model_response"]]
    print(f"Yanlis siniflandirilan (hatasiz cagri icinde) vaka sayisi: {len(df_incorrect)}")
else:
    print("Henuz sonuc yok.")
    df_incorrect = pd.DataFrame()

df_incorrect


In [ ]:
# E03 vs E04 Order Sensitivity: ayni vakalarda (ayni case_id, ayni repetition)
# model karari degisti mi? Accuracy ayni kalsa bile belirli vakalarda
# tahminler degismis olabilir; bu nedenle genel accuracy yerine vaka
# bazli karsilastirma yapiyoruz.
if not df_results.empty:
    e03 = df_results[df_results["experiment"] == "E03_diverse_few_shot"][
        ["case_id", "repetition", "expected_label", "parsed_label"]
    ].rename(columns={"parsed_label": "e03_label"})
    e04 = df_results[df_results["experiment"] == "E04_reversed_order"][
        ["case_id", "repetition", "parsed_label"]
    ].rename(columns={"parsed_label": "e04_label"})

    df_order_sensitivity = e03.merge(e04, on=["case_id", "repetition"], how="inner")
    df_order_sensitivity["decision_changed"] = df_order_sensitivity["e03_label"] != df_order_sensitivity["e04_label"]

    n_changed = int(df_order_sensitivity["decision_changed"].sum())
    print(f"E03 ile E04 arasinda karari degisen vaka sayisi: {n_changed} / {len(df_order_sensitivity)}")
    df_order_sensitivity_changed = df_order_sensitivity[df_order_sensitivity["decision_changed"]]
else:
    print("Henuz sonuc yok.")
    df_order_sensitivity = pd.DataFrame()
    df_order_sensitivity_changed = pd.DataFrame()

df_order_sensitivity_changed


## Step 11 — Visualization

Bes basit grafikle deneyleri karsilastiriyoruz: Label Accuracy, Format
Compliance Rate, Average Token Usage, Average Latency ve Per-Category
Accuracy. Egitim amacli oldugu icin kasitli olarak sade tutulmustur;
seaborn veya ek bir gorsellestirme kutuphanesi eklenmemistir.


In [ ]:
if not df_comparison.empty:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))

    axes[0, 0].bar(df_comparison["experiment"], df_comparison["label_accuracy"])
    axes[0, 0].set_title("Label Accuracy by Experiment")
    axes[0, 0].set_ylabel("Accuracy")
    axes[0, 0].tick_params(axis="x", rotation=45)

    axes[0, 1].bar(df_comparison["experiment"], df_comparison["format_compliance_rate"])
    axes[0, 1].set_title("Format Compliance Rate by Experiment")
    axes[0, 1].set_ylabel("Compliance Rate")
    axes[0, 1].tick_params(axis="x", rotation=45)

    axes[0, 2].bar(df_comparison["experiment"], df_comparison["avg_prompt_tokens"], label="Prompt Tokens")
    axes[0, 2].bar(
        df_comparison["experiment"],
        df_comparison["avg_completion_tokens"],
        bottom=df_comparison["avg_prompt_tokens"],
        label="Completion Tokens",
    )
    axes[0, 2].set_title("Average Token Usage by Experiment")
    axes[0, 2].set_ylabel("Tokens")
    axes[0, 2].tick_params(axis="x", rotation=45)
    axes[0, 2].legend()

    axes[1, 0].bar(df_comparison["experiment"], df_comparison["avg_latency_ms"])
    axes[1, 0].set_title("Average API Latency by Experiment")
    axes[1, 0].set_ylabel("Latency (ms)")
    axes[1, 0].tick_params(axis="x", rotation=45)

    # Per-Category Accuracy: her kategori icin, her deneyin accuracy degerini
    # gruplanmis cubuklarla gosteriyoruz.
    ax = axes[1, 1]
    n_categories = len(df_per_category_accuracy.columns)
    n_experiments = len(df_per_category_accuracy.index)
    bar_width = 0.8 / n_experiments
    x_positions = range(n_categories)
    for i, experiment_key in enumerate(df_per_category_accuracy.index):
        offsets = [x + i * bar_width for x in x_positions]
        ax.bar(offsets, df_per_category_accuracy.loc[experiment_key], width=bar_width, label=experiment_key)
    ax.set_xticks([x + bar_width * (n_experiments - 1) / 2 for x in x_positions])
    ax.set_xticklabels(df_per_category_accuracy.columns, rotation=45, ha="right")
    ax.set_title("Per-Category Accuracy by Experiment")
    ax.set_ylabel("Accuracy")
    ax.legend(fontsize=7)

    axes[1, 2].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Gorsellestirme icin once Step 8'de RUN_LIVE = True yaparak deneyleri calistirin.")


In [ ]:
# E03 vs E04 siralama karsilastirma tablosu (kucuk pandas tablosu).
if not df_order_sensitivity.empty:
    df_order_summary = pd.DataFrame({
        "metric": ["n_cases_compared", "n_decisions_changed", "decision_change_rate"],
        "value": [
            len(df_order_sensitivity),
            int(df_order_sensitivity["decision_changed"].sum()),
            df_order_sensitivity["decision_changed"].mean(),
        ],
    })
else:
    print("Henuz sonuc yok.")
    df_order_summary = pd.DataFrame()

df_order_summary


## Step 12 — Discussion & Engineering Conclusions

Yukaridaki sonuclari calistirdiktan sonra asagidaki sorulari `df_comparison`,
`df_per_category_accuracy` ve `df_order_sensitivity` tablolarina bakarak
tartisin. Olcum yapilmadan (yani `RUN_LIVE = True` ile deneyler
calistirilmadan) bu sorulara verilecek cevaplar sadece spekulasyondur —
sonuc uydurmayin.

1. `E01_zero_shot` hangi kategorilerde yeterli dogruluga ulasti, hangilerinde
   zorlandi?
2. `E02_redundant_few_shot` ile `E03_diverse_few_shot` arasinda
   `label_accuracy` ve per-category accuracy nasil farklilasti?
3. `E03_diverse_few_shot` ile `E04_reversed_order` arasinda genel accuracy
   degismese bile, hangi vakalarda model karari degisti
   (`df_order_sensitivity_changed`)?
4. Hangi deney en fazla prompt token kullandi? Bu, daha yuksek accuracy ile
   sonuclandi mi, yoksa sadece maliyeti artirdi mi?
5. Invalid Response Rate ve Format Compliance Rate deneyler arasinda nasil
   degisti?
6. Giris bolumundeki 5 arastirma sorusu, gercek olculerle ne derece
   cevaplanabildi?

## Yontembilimsel Sinirlamalar

- Few-Shot, her zaman Zero-Shot'tan daha iyi sonuc vermez.
- More Examples, her zaman Higher Accuracy anlamina gelmez.
- Example Quality ve Example Diversity farkli kavramlardir; bu lab'de tum
  ornekler dogru etiketlidir (quality sabit), yalnizca diversity/selection
  degisir.
- `E02` ile `E03` arasinda birden fazla ornek ozelligi (icerik, temsil
  gucu) birlikte degistigi icin tek bir nedensel etki izole edilmez.
- `E03` ile `E04` siralama karsilastirmasi ayni orneklerle yapilir, ancak
  tek calistirma order sensitivity hakkinda kesin bir sonuc vermez.
- `temperature = 0` tamamen deterministik davranisi garanti etmez.
- Kucuk evaluation dataset (20 kayit, deney basina 10 vaka) ile genel
  performans iddialari olusturulamaz; sonuclar kesfedici ve egitim
  amaclidir.
- In-Context Learning, bu deneyde model agirliklarinin guncellendigi bir
  fine-tuning islemi DEGILDIR; ornekler yalnizca o tek istek icin baglam
  olarak sunulur.
- Token maliyeti ve model dogrulugu BIRLIKTE degerlendirilmelidir; yuksek
  token kullanimi tek basina bir basari gostergesi degildir.
